<a href="https://colab.research.google.com/github/raz0208/Natural-Language-Processing-Practices/blob/main/TopicModelling/EmbeddingsAnalysis_TopicClustering_ModernBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Topic Clustring

### Clustering Topic Models from TurfTopic

1. TurfTopic Default model and configuration
2. Use ModernBERT for embeddings

In [ ]:
# Install libraries and packages
!pip install 'turftopic[umap-learn, datamapplot]'

In [2]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch

In [4]:
# Read and Load dataset
dataset = pd.read_csv('sample_PubMedDataAbstracts.csv')

# Show the datasets
### Abstract Embeddings Sample Dataset
print('Node Content:', dataset.shape)
print(dataset)

Node Content: (10000, 4)
      Unnamed: 0                                              title  \
0              0  Phenotypic variability of Niemann-Pick disease...   
1              1  Recurrent hypoglycemia secondary to metformin ...   
2              2  Adaptation of the Ambulatory and Home Care Rec...   
3              3  Multidimensional family therapy in adolescents...   
4              4  Balanced crystalloids versus isotonic saline i...   
...          ...                                                ...   
9995        9995  Methylmercury in Industrial Harbor Sediments i...   
9996        9996  Factors Affecting Secondhand Smoke Avoidance B...   
9997        9997  Predicting Infectious Disease Using Deep Learn...   
9998        9998  Diosgenin Glucoside Protects against Spinal Co...   
9999        9999  Omics Approaches for Engineering Wheat Product...   

                                               abstract  year  
0     Background Niemann-Pick disease type C (NPC) i...  2

In [5]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  10000 non-null  int64 
 1   title       10000 non-null  object
 2   abstract    10000 non-null  object
 3   year        10000 non-null  object
dtypes: int64(1), object(3)
memory usage: 312.6+ KB


In [6]:
# Extract only the 'abstract' column and drop others
abstracts = dataset['abstract'].tolist()

# Display a few samples to verify
abstracts[:5]

['Background Niemann-Pick disease type C (NPC) is a lysosomal storage disorder with severe prognosis. Disease-specific therapy is crucial to prevent disease progression; however, diagnosing NPC is quite difficult because of remarkably variable clinical presentations. The NPC Suspicion Index (NPC-SI) was developed to overcome this problem. Identifying preclinical cases is important for prevention and therapy. Here, we report three newly diagnosed NPC cases, one typical juvenile-onset case and the cases of two sisters with symptoms neurologically/psychiatrically indistinguishable from dystonia and schizophrenia, respectively. Case presentation In Case 1, a 25-year-old man presented with a 14-year history of intellectual disability, clumsiness, spastic ataxia, dysphagia, and frequent falls. Neurological examination revealed vertical supranuclear gaze palsy and involuntary movements. Ultrasonography revealed mild splenomegaly, and filipin staining of skin fibroblasts was positive with a va

## **1. TurfTopic Default model and configuration**
  - Use "ModernBERT" to extract embeddings
  - Use Top2Vec to topic modelling ans clustering

In [7]:
# Import topic clustring required libraries
from sentence_transformers import SentenceTransformer
from turftopic import Top2Vec
from turftopic import BERTopic
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [ ]:
# Load ModernBERT tokenizer and model from Hugging Face
MODEL_NAME = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

### Force to use GPU for embeddings

In [9]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Function to get embeddings for a list of texts
def get_embeddings(texts, tokenizer, model):
    model.eval()
    device = next(model.parameters()).device  # ensures model and inputs are on same device
    embeddings = []

    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=False, padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}  # move inputs to GPU
            outputs = model(**inputs)
            cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()  # back to CPU
            embeddings.append(cls_embedding)

    return embeddings

### Embeddings use by CPU

In [ ]:
# # Function to get embeddings for a list of texts
# def get_embeddings(texts, tokenizer, model):
#     model.eval()
#     embeddings = []

#     with torch.no_grad():
#         for text in texts:
#             inputs = tokenizer(text, return_tensors="pt", truncation=False, padding=True)
#             outputs = model(**inputs)
#             # Use [CLS] token embedding (first token)
#             cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
#             embeddings.append(cls_embedding)

#     return embeddings

In [10]:
# Wrap the texts with tqdm for progress visualization
abstracts = abstracts
abstracts_with_progress = tqdm(abstracts, desc="Embedding abstracts")

# Call the function with tqdm-wrapped list
abstract_embeddings = get_embeddings(abstracts_with_progress, tokenizer, model)

Embedding abstracts: 100%|██████████| 10000/10000 [06:48<00:00, 24.49it/s]


In [ ]:
# # save abstract_embeddings into csv file
# np.savetxt("abstract_embeddings.csv", abstract_embeddings, delimiter=",")

### Use Pre-embeddings csv file

In [ ]:
# # Save the embedding to a csv file
# embedding_df = pd.DataFrame(embeddings)
# embedding_df.to_csv('abstract_embeddings.csv', index=False)

# # Show first 10 embeddings
# embeddings[:10]

In [ ]:
# # get a sample
# sample = embeddings[0]

# sample.shape

### Continue to Topic Modelling

In [11]:
# Show shape of the first embedding
len(abstract_embeddings), abstract_embeddings[0].shape

(10000, (768,))

In [12]:
# Show embeddings matrix and Check the dimention of each eambeding
embeddings = np.array(abstract_embeddings)
print(embeddings,"\n\n")

[[ 0.30497697 -0.20869878 -0.18874036 ... -1.1213119   0.63652396
  -0.5493275 ]
 [ 0.4623834  -0.6523201   0.29970372 ... -1.2555125   1.128265
  -0.3443874 ]
 [-0.23266809 -0.51089746 -0.0102424  ... -1.5995301   0.7679716
  -0.77237844]
 ...
 [-0.2716385  -0.38257405 -0.21645126 ... -1.5033063   0.93543106
  -0.24376546]
 [ 0.09230014 -0.67846876 -0.27810726 ... -1.7891287   0.487746
  -0.6473106 ]
 [-0.08118434 -0.49155283 -0.31141263 ... -1.7697753   0.9467157
  -0.40294534]] 




In [ ]:
# Training model (Uses HDBSCAN and umap)
model = Top2Vec(encoder="answerdotai/ModernBERT-base", random_state=42)
#topics = model.fit_transform(abstracts, embeddings=embeddings)
topic_data = model.prepare_topic_data(abstracts, embeddings=embeddings)

In [14]:
topic_data

TopicData
├── corpus (10000)
├── vocab (10027,)
├── document_term_matrix (10000, 10027)
├── topic_term_matrix (6, 10027)
├── document_topic_matrix (10000, 6)
├── document_representation (10000, 768)
├── transform
├── topic_names (6)
├── has_negative_side
└── hierarchy

In [15]:
model.print_topics()

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Highest Ranking                                                                                      ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       -1 │ affymetrix, oncogenes, bronchiectasis, immunoassays, assessments, sagittal, enhancer, amygdala,      │
│          │ carotenoids, oncogenic                                                                               │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │ slightly, assessments, etching, gg, mann, oncogenes, lysine, lysosomes, shed, sagittal               │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │ assessments, affymetrix, etching, oncogenes, immunoassays, germplasm, gg, lysosomes, sagittal, mann  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │ assessments, etching, affymetrix, amygdala, sirnas, sagittal, oncogenes, mann, lysosomes, hepatoma   │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        3 │ assessments, oncogenes, etching, slightly, affymetrix, gg, mann, achievable, lysosomes, sagittal     │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        4 │ assessments, retinopathy, oncogenes, etching, lysosomes, affymetrix, suggests, sagittal, sirnas,     │
│          │ proteinuria                                                                                          │
└──────────┴──────────────────────────────────────────────────────────────────────────────────────────────────────┘

In [16]:
# Cluster model hierarchy
model.hierarchy.cut(3).plot_tree()

In [17]:
# Merging topics to reduce number of topics
model.reduce_topics(n_reduce_to=25)
print(model.hierarchy.cut(3))

/usr/local/lib/python3.11/dist-packages/turftopic/models/_hierarchical_clusters.py:276: UserWarning:

Number of clusters is already 5 <= 25, nothing to do.



Root: 
├── -1: affymetrix, oncogenes, bronchiectasis, immunoassays, assessments, sagittal, enhancer, amygdala,
│   carotenoids, 
│   oncogenic
├── 0: slightly, assessments, etching, gg, mann, oncogenes, lysine, lysosomes, shed, sagittal
├── 1: assessments, affymetrix, etching, oncogenes, immunoassays, germplasm, gg, lysosomes, sagittal,
│   mann
├── 2: assessments, etching, affymetrix, amygdala, sirnas, sagittal, oncogenes, mann, lysosomes, 
│   hepatoma
├── 3: assessments, oncogenes, etching, slightly, affymetrix, gg, mann, achievable, lysosomes, 
│   sagittal
└── 4: assessments, retinopathy, oncogenes, etching, lysosomes, affymetrix, suggests, sagittal, sirnas,
    proteinuria



In [ ]:
# # Model hierarchy after merging topics
# fig = model.hierarchy[156].plot_tree()
# fig.show()

In [18]:
# We will reset the hierarchy, so that we can see all topics at once.
model.reset_topics()
fig = model.plot_clusters_datamapplot(hover_text=dataset["title"])
fig.show()

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_data.py:258: UserWarning:

Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.



## **Evaluation**
  - Silhoot-Score

In [20]:
# Show embeddings matrix and Check the dimention of each eambeding
embeddings1 = np.array(abstract_embeddings)
print(embeddings1,"\n\n")

[[ 0.30497697 -0.20869878 -0.18874036 ... -1.1213119   0.63652396
  -0.5493275 ]
 [ 0.4623834  -0.6523201   0.29970372 ... -1.2555125   1.128265
  -0.3443874 ]
 [-0.23266809 -0.51089746 -0.0102424  ... -1.5995301   0.7679716
  -0.77237844]
 ...
 [-0.2716385  -0.38257405 -0.21645126 ... -1.5033063   0.93543106
  -0.24376546]
 [ 0.09230014 -0.67846876 -0.27810726 ... -1.7891287   0.487746
  -0.6473106 ]
 [-0.08118434 -0.49155283 -0.31141263 ... -1.7697753   0.9467157
  -0.40294534]] 




In [ ]:
# Training model (Uses HDBSCAN and umap)
model1 = BERTopic(encoder="answerdotai/ModernBERT-base", random_state=42)
#topics = model1.fit_transform(abstracts, embeddings=embeddings1)
topic_data1 = model1.prepare_topic_data(abstracts, embeddings=embeddings1)

In [22]:
topic_data1

TopicData
├── corpus (10000)
├── vocab (10027,)
├── document_term_matrix (10000, 10027)
├── topic_term_matrix (7, 10027)
├── document_topic_matrix (10000, 7)
├── document_representation (10000, 768)
├── transform
├── topic_names (7)
├── has_negative_side
└── hierarchy

In [23]:
model1.print_topics()

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Highest Ranking                                                                        ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       -1 │ egfr, cc, strain, pic, genus, 05, diagnosis, patient, species, thrombin                │
├──────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │ patients, study, results, methods, background, health, group, using, conclusions, data │
├──────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │ des, une, les, la, le, dans, et, en, cas, est                                          │
├──────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │ electron, metal, reaction, gender, fl, pt, lewis, electrochemical, bank, bond          │
├──────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│        3 │ patients, study, ich, heat, results, mortality, background, type, methods, age         │
├──────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│        4 │ cells, cell, study, using, results, high, based, expression, activity, used            │
├──────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│        5 │ cells, patients, hiv, pod, reactivity, men, tick, results, group, tidal                │
└──────────┴────────────────────────────────────────────────────────────────────────────────────────┘

In [24]:
# Cluster model hierarchy
model1.hierarchy.cut(3).plot_tree()

In [25]:
# Merging topics to reduce number of topics
model1.reduce_topics(n_reduce_to=25)
print(model1.hierarchy.cut(3))

/usr/local/lib/python3.11/dist-packages/turftopic/models/_hierarchical_clusters.py:276: UserWarning:

Number of clusters is already 6 <= 25, nothing to do.



Root: 
├── -1: egfr, cc, strain, pic, genus, 05, diagnosis, patient, species, thrombin
├── 0: patients, study, results, methods, background, health, group, using, conclusions, data
├── 1: des, une, les, la, le, dans, et, en, cas, est
├── 2: electron, metal, reaction, gender, fl, pt, lewis, electrochemical, bank, bond
├── 3: patients, study, ich, heat, results, mortality, background, type, methods, age
├── 4: cells, cell, study, using, results, high, based, expression, activity, used
└── 5: cells, patients, hiv, pod, reactivity, men, tick, results, group, tidal

